# Credit Union Quarterly Data IngestionThis notebook executes the full-coverage ingestion pipeline in `ingest_ncua_call_report.py`.

In [0]:
# Configure your ingestion window (inclusive).# Example: start_year=1994 and end_year=2026 requests all quarters in that range.start_year = 1994end_year = 2026output_file = "NCUA_Call_Report.csv"multirecord_output_file = "NCUA_Call_Report_Multirecord.csv"run_ingestion(    start_year=start_year,    end_year=end_year,    output_file=output_file,    multirecord_output_file=multirecord_output_file,    cleanup_temp_files=True,)

In [0]:
from ingest_ncua_call_report import (
    run_ingestion,
    discover_quarter_urls,
    build_requested_quarters,
)
from pyspark.sql.functions import to_date, col
import os
import re
import gc
import ssl

# ---------------------------------------------------------------------------
# Suppress verbose JVM stacktraces
# ---------------------------------------------------------------------------
spark.sparkContext.setLogLevel("ERROR")

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
START_YEAR = 1994
END_YEAR = 2026
SCHEMA = "workspace.credituniontest"
MAIN_TABLE = f"{SCHEMA}.ncua_call_report"
MULTI_TABLE = f"{SCHEMA}.ncua_call_report_multirecord"
MAPPING_TABLE = f"{SCHEMA}.ncua_column_mapping"
WORK_DIR = os.getcwd()

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA}")

# ---------------------------------------------------------------------------
# Discover quarter URLs once upfront (avoids 32 identical HTTP calls)
# ---------------------------------------------------------------------------
ssl_ctx = ssl.create_default_context()
try:
    quarter_urls = discover_quarter_urls(ssl_ctx)
    print(f"Discovered {len(quarter_urls)} quarter links from NCUA page.")
except Exception as e:
    quarter_urls = {}
    print(f"WARN: Could not discover quarter links ({e}). Using fallback URLs.")

# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------
def sanitize_column_name(header: str) -> str:
    """Lightweight cleanup for raw column names from the ingestion output.
    With raw_column_names=True, columns are already clean codes like
    CU_NUMBER, ACCT_881, ACCT_010__FS220A. Just normalize edge cases."""
    clean = re.sub(r"[^a-zA-Z0-9_]", "_", header.strip())
    clean = re.sub(r"_+", "_", clean).strip("_").upper()
    return clean


def bulk_rename_columns(df, column_mapping_accum):
    """Rename all columns in one shot via toDF() instead of per-column loop."""
    original_names = df.columns
    new_names = []
    seen = {}
    for orig in original_names:
        clean = sanitize_column_name(orig)
        if clean in seen:
            seen[clean] += 1
            clean = f"{clean}_{seen[clean]}"
        else:
            seen[clean] = 0
        new_names.append(clean)
        # We only need mapping entries for account columns, not dupes
        if seen[clean] == 0:
            column_mapping_accum[clean] = orig

    return df.toDF(*new_names)


def add_report_date(df):
    """Parse CYCLE_DATE (numeric YYYYMMDD) into a proper DATE column."""
    if "CYCLE_DATE" in df.columns:
        df = df.withColumn(
            "report_date",
            to_date(col("CYCLE_DATE").cast("string"), "yyyyMMdd"),
        )
    return df


def abs_path(filename: str) -> str:
    """Return file:// absolute path that Spark can resolve on the driver."""
    return "file:" + os.path.join(WORK_DIR, filename)


def cleanup_files(*paths):
    for p in paths:
        try:
            full = os.path.join(WORK_DIR, p) if not os.path.isabs(p) else p
            if os.path.exists(full):
                os.remove(full)
        except OSError:
            pass

# ---------------------------------------------------------------------------
# Year-by-year ingestion loop
# ---------------------------------------------------------------------------
global_account_map = {}      # code -> description, accumulated across years
column_name_tracking = {}    # clean_name -> original_header
first_main_write = True
first_multi_write = True
years_ok = []
years_failed = []

for year in range(START_YEAR, END_YEAR + 1):
    output_csv = f"ncua_main_{year}.csv"
    multi_csv = f"ncua_multi_{year}.csv"

    print(f"\n{'='*60}")
    print(f"Processing year {year}")
    print(f"{'='*60}")

    try:
        account_map = run_ingestion(
            start_year=year,
            end_year=year,
            output_file=output_csv,
            multirecord_output_file=multi_csv,
            cleanup_temp_files=True,
            raw_column_names=True,
        )

        # Accumulate the authoritative account code -> description mapping
        if account_map:
            global_account_map.update(account_map)

        # -- Main table --
        local_path = os.path.join(WORK_DIR, output_csv)
        if os.path.exists(local_path) and os.path.getsize(local_path) > 0:
            df = spark.read.csv(abs_path(output_csv), header=True, inferSchema=True)
            df = bulk_rename_columns(df, column_name_tracking)
            df = add_report_date(df)

            write_mode = "overwrite" if first_main_write else "append"
            (
                df.write
                .mode(write_mode)
                .option("mergeSchema", "true")
                .saveAsTable(MAIN_TABLE)
            )
            row_count = df.count()
            print(f"  -> {row_count:,} rows written to {MAIN_TABLE} ({write_mode})")
            first_main_write = False
            del df
        else:
            print(f"  -> No main data for {year}, skipping.")

        # -- Multirecord table --
        multi_local = os.path.join(WORK_DIR, multi_csv)
        if os.path.exists(multi_local) and os.path.getsize(multi_local) > 0:
            multi_df = spark.read.csv(abs_path(multi_csv), header=True, inferSchema=True)
            multi_df = bulk_rename_columns(multi_df, column_name_tracking)
            multi_df = add_report_date(multi_df)

            multi_mode = "overwrite" if first_multi_write else "append"
            multi_df.write.mode(multi_mode).option("mergeSchema", "true").saveAsTable(MULTI_TABLE)
            print(f"  -> Multirecord rows written to {MULTI_TABLE} ({multi_mode})")
            first_multi_write = False
            del multi_df

        years_ok.append(year)

    except Exception as e:
        print(f"  -> FAILED: {type(e).__name__}: {e}")
        years_failed.append((year, str(e)))

    finally:
        cleanup_files(output_csv, multi_csv)
        gc.collect()

# ---------------------------------------------------------------------------
# Column mapping reference table
# Built from the authoritative AcctDesc.txt data returned by run_ingestion,
# not from re-parsing header strings.
# ---------------------------------------------------------------------------
if global_account_map:
    mapping_rows = [
        (code, sanitize_column_name(code), description)
        for code, description in global_account_map.items()
    ]
    mapping_df = spark.createDataFrame(
        mapping_rows, ["account_code", "column_name", "description"]
    )
    mapping_df.write.mode("overwrite").saveAsTable(MAPPING_TABLE)
    print(f"\nColumn mapping saved to {MAPPING_TABLE} ({len(mapping_rows)} entries)")

# ---------------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------------
print(f"\n{'='*60}")
print(f"COMPLETE: {len(years_ok)} years succeeded, {len(years_failed)} failed")
if years_failed:
    print("Failed years:")
    for y, reason in years_failed:
        print(f"  {y}: {reason}")

if not first_main_write:
    total = spark.table(MAIN_TABLE).count()
    cols = len(spark.table(MAIN_TABLE).columns)
    print(f"Final table: {total:,} rows x {cols} columns in {MAIN_TABLE}")